# Image ⇄ Transcript CLIP Training Notebook
This notebook is a refactored version of the original `main.py`, `model.py`, and loss definitions you shared. Run each cell sequentially.

Feel free to tweak paths or hyper‑parameters for quick experiments.

In [11]:

!pwd

/endosome/archive/DPDS/Xiao_lab/shared/jia_yao/Image2Transcript


In [25]:
import os, torch, math, numpy as np
import scanpy as sc
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import NearestNeighbors
from torch.cuda.amp import GradScaler, autocast
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models.vision_transformer import vit_b_16

# ---------------- CONFIG ----------------
gene_dir = "data/demo/gene_expression"          # 🖉 change as needed
img_dir  = "data/demo/images"          # 🖉 change as needed
out_dir  = "output_zinb"        # 🖉 change as needed
batch_size = 16
epochs      = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('💻 device:', device)


💻 device: cpu


In [26]:
from tqdm.auto import tqdm
from sklearn.neighbors import NearestNeighbors

class XeniumCellDataset(Dataset):
    """Creates (image, gene features) pairs and caches neighbor logFC."""
    def __init__(self, gene_dir, img_dir, transform=None):
        self.gene_dir = gene_dir
        self.img_dir  = img_dir
        self.transform = transform or transforms.ToTensor()

        self.data_pairs, self.shared_genes = [], None
        self.cell_exprs, self.cell_coords = [], []
        self.slide_ids = []                 # for future group split
        self._prepare()

    def _prepare(self):
        expr_files = sorted([f for f in os.listdir(self.gene_dir) if f.endswith('.h5ad')])
        if not expr_files:
            raise RuntimeError(f'No .h5ad under {self.gene_dir}')

        # ① 读取每个 .h5ad（带进度条）
        tmp, shared = [], None
        for fn in tqdm(expr_files, desc='🔍 Reading h5ad'):
            ad = sc.read_h5ad(os.path.join(self.gene_dir, fn))
            shared = set(ad.var_names) if shared is None else shared & set(ad.var_names)
            tmp.append((fn, ad))
        self.shared_genes = sorted(shared)
        print(f'✅ Shared genes: {len(self.shared_genes)}')

        # ② 遍历细胞、检查图像
        for fn, ad in tqdm(tmp, desc='🖼️  Collecting image-gene pairs'):
            img_folder = os.path.join(self.img_dir, fn.replace('.h5ad', ''))
            if not os.path.isdir(img_folder):
                continue
            ad = ad[:, self.shared_genes]
            X = ad.X.toarray() if hasattr(ad.X, 'toarray') else ad.X
            coords = ad.obs[['x_centroid', 'y_centroid']].values

            slide_id = os.path.splitext(fn)[0]
            for i, cell_id in enumerate(ad.obs_names):
                img_fp = os.path.join(img_folder, f'{cell_id}.png')
                if os.path.exists(img_fp):
                    self.data_pairs.append((img_fp, X[i], coords[i], slide_id))
                    self.slide_ids.append(slide_id)

            self.cell_exprs.append(X)
            self.cell_coords.append(coords)

        # ③ 连接并建立 KNN
        if not self.cell_exprs:
            raise RuntimeError('❌ No matching image-gene pairs found.')
        self.cell_exprs  = np.concatenate(self.cell_exprs, axis=0)
        self.cell_coords = np.concatenate(self.cell_coords, axis=0)
        self.nn_model = NearestNeighbors(n_neighbors=6).fit(self.cell_coords)

    def __len__(self): 
        return len(self.data_pairs)

    def __getitem__(self, idx):
        img_fp, expr, coord, _ = self.data_pairs[idx]
        img = Image.open(img_fp).convert('RGB')
        img = self.transform(img)

        # neighbour logFC
        _, ind = self.nn_model.kneighbors([coord])
        neigh = self.cell_exprs[ind[0][1:]]
        neigh_mean = neigh.mean(axis=0)
        logfc = np.log2((expr + 1e-3) / (neigh_mean + 1e-3))

        feat = np.concatenate([expr, logfc])
        return img, torch.tensor(feat, dtype=torch.float32)


/archive/DPDS/Xiao_lab/shared/jia_yao/envs/image2transcripts/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [27]:
def zinb_nll(x, mu, theta, pi, eps=1e-8):
    """Negative log‑likelihood of ZINB."""
    log_theta_mu_eps = torch.log(theta + mu + eps)
    nb_case = (
        torch.lgamma(theta + x)
        - torch.lgamma(theta)
        - torch.lgamma(x + 1)
        + theta * (torch.log(theta + eps) - log_theta_mu_eps)
        + x * (torch.log(mu + eps) - log_theta_mu_eps)
    )
    zero_mask = (x < 1e-8).float()
    nll = -torch.where(
        zero_mask.bool(),
        torch.log(pi + (1.0 - pi) * torch.exp(nb_case) + eps),
        torch.log(1.0 - pi + eps) + nb_case,
    )
    return nll.mean()


def full_loss(i_emb, g_emb, zinb_params, gene_input, t_img, t_gen,
              w_align=0.1, w_zinb=0.5):
    B = i_emb.size(0)
    logits_ig = i_emb @ g_emb.T / t_img
    logits_gi = g_emb @ i_emb.T / t_gen
    lbl = torch.arange(B, device=i_emb.device)
    c_loss = (F.cross_entropy(logits_ig, lbl) + F.cross_entropy(logits_gi, lbl)) / 2
    a_loss = F.mse_loss(i_emb, g_emb)

    G = gene_input.size(1) // 2
    x_raw = gene_input[:, :G]
    mu, theta, pi = zinb_params
    z_loss = zinb_nll(x_raw, mu, theta, pi)
    total = c_loss + w_align * a_loss + w_zinb * z_loss
    return total, c_loss.item(), a_loss.item(), z_loss.item()


In [28]:
class GeneTransformerEncoder(nn.Module):
    def __init__(self, gene_dim: int, embed_dim: int = 768, heads: int = 4):
        super().__init__()
        self.value_proj = nn.Linear(2, embed_dim, bias=False)
        self.gene_id_emb = nn.Embedding(gene_dim, embed_dim)
        self.pos_emb = nn.Parameter(torch.randn(1, gene_dim, embed_dim))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=heads, dim_feedforward=1024, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=2)

    def forward(self, x):
        B, D = x.shape
        G = D // 2
        x = x.view(B, G, 2)
        tok = (
            self.value_proj(x)
            + self.gene_id_emb.weight[:G]
            + self.pos_emb[:, :G, :]
        )
        tok = self.transformer(tok)
        return tok.mean(dim=1)


class Image2Transcripts(nn.Module):
    def __init__(self, gene_dim: int, embed_dim: int = 768, vit_weights="IMAGENET1K_V1"):
        super().__init__()
        self.image_encoder = vit_b_16(weights=vit_weights)
        self.image_encoder.heads = nn.Identity()
        self.image_proj = nn.Linear(768, embed_dim, bias=False)

        self.gene_encoder = GeneTransformerEncoder(gene_dim, embed_dim)
        self.gene_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        self.zinb_head = nn.Linear(embed_dim, gene_dim * 3)
        self.t_img = nn.Parameter(torch.tensor(0.07))
        self.t_gen = nn.Parameter(torch.tensor(0.07))
        self.gene_dim = gene_dim

    @staticmethod
    def _softplus(x): return F.softplus(x) + 1e-4

    def _split_zinb(self, z):
        B = z.size(0)
        z = z.view(B, 3, self.gene_dim)
        mu = self._softplus(z[:, 0])
        theta = self._softplus(z[:, 1])
        pi = torch.sigmoid(z[:, 2])
        return mu, theta, pi

    def forward(self, image, gene_input):
        i_emb = F.normalize(self.image_proj(self.image_encoder(image)), dim=-1)
        g_emb_raw = self.gene_encoder(gene_input)
        g_emb = F.normalize(self.gene_proj(g_emb_raw), dim=-1)
        mu, theta, pi = self._split_zinb(self.zinb_head(g_emb_raw))
        return i_emb, g_emb, (mu, theta, pi)


In [29]:
def train(model, train_loader, val_loader, device, epochs, out_dir,
          lr=3e-4, warm_epochs=3):
    os.makedirs(out_dir, exist_ok=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=epochs, T_mult=1)
    scaler = GradScaler()
    best = float('inf')

    log_fp = os.path.join(out_dir, 'train_log.csv')
    with open(log_fp, 'w') as f:
        f.write('epoch,train_loss,val_loss,train_c,val_c,train_a,val_a,train_z,val_z\n')

    for ep in range(1, epochs + 1):
        # ---- train ----
        model.train()
        tr_metrics = np.zeros(4)
        for img, g in train_loader:
            img, g = img.to(device), g.to(device)
            opt.zero_grad(set_to_none=True)
            with autocast():
                i_emb, g_emb, zinb_pars = model(img, g)
                loss, c, a, z = full_loss(i_emb, g_emb, zinb_pars, g, model.t_img, model.t_gen)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tr_metrics += np.array([loss.item(), c, a, z])
        scheduler.step()

        # ---- val ----
        model.eval()
        vl_metrics = np.zeros(4)
        with torch.no_grad(), autocast():
            for img, g in val_loader:
                img, g = img.to(device), g.to(device)
                i_emb, g_emb, zinb_pars = model(img, g)
                loss, c, a, z = full_loss(i_emb, g_emb, zinb_pars, g, model.t_img, model.t_gen)
                vl_metrics += np.array([loss.item(), c, a, z])

        tr_metrics /= len(train_loader)
        vl_metrics /= len(val_loader)
        with open(log_fp, 'a') as f:
            f.write(f"{ep},{tr_metrics[0]:.4f},{vl_metrics[0]:.4f},{tr_metrics[1]:.4f},{vl_metrics[1]:.4f},"
                    f"{tr_metrics[2]:.4f},{vl_metrics[2]:.4f},{tr_metrics[3]:.4f},{vl_metrics[3]:.4f}\n")

        print(f"E{ep:03d}: L={tr_metrics[0]:.3f}/{vl_metrics[0]:.3f}  "
              f"C={tr_metrics[1]:.3f}/{vl_metrics[1]:.3f}  "
              f"A={tr_metrics[2]:.3f}/{vl_metrics[2]:.3f}  "
              f"Z={tr_metrics[3]:.3f}/{vl_metrics[3]:.3f}")

        if vl_metrics[0] < best:
            best = vl_metrics[0]
            torch.save(model.state_dict(), os.path.join(out_dir, 'best_model.pt'))
            print('   ✔️  saved best')


In [30]:
tfm = transforms.Compose([
transforms.Resize((224, 224)),
transforms.RandomHorizontalFlip(),
transforms.ColorJitter(hue=.05, saturation=.05),
transforms.ToTensor(),
])


full_ds = XeniumCellDataset(gene_dir, img_dir, transform=tfm)


train_len = int(0.8 * len(full_ds))
val_len = len(full_ds) - train_len
tr_ds, vl_ds = torch.utils.data.random_split(
full_ds, [train_len, val_len],
generator=torch.Generator().manual_seed(42) # 可复现
)

tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,
num_workers=8, pin_memory=True)
vl_ld = DataLoader(vl_ds, batch_size=batch_size, shuffle=False,
num_workers=8, pin_memory=True)


model = Image2Transcripts(gene_dim=len(full_ds.shared_genes)).to(device)
train(model, tr_ld, vl_ld, device, epochs, out_dir)

🔍 Reading h5ad:   0%|          | 0/1 [00:00<?, ?it/s]

🔍 Reading h5ad: 100%|██████████| 1/1 [00:00<00:00, 26.59it/s]


✅ Shared genes: 372


🖼️  Collecting image-gene pairs: 100%|██████████| 1/1 [00:04<00:00,  4.12s/it]
/tmp/ipykernel_25300/1238062884.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/archive/DPDS/Xiao_lab/shared/jia_yao/envs/image2transcripts/lib/python3.9/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
/tmp/ipykernel_25300/1238062884.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/archive/DPDS/Xiao_lab/shared/jia_yao/envs/image2transcripts/lib/python3.9/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


KeyboardInterrupt: 

In [35]:

# ───────────────────────────────────────────────────────────
#  main_fast_groupfold.py — ViT ⇄ Gene CLIP (GroupKFold split)
#  • fast dataloader (read_image + pre‑computed logFC)
#  • GroupKFold (slide‑level) for train / val
# ───────────────────────────────────────────────────────────
import os, torch, numpy as np, scanpy as sc
from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.io import read_image
import torchvision.transforms.v2 as T2
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import GroupKFold
from torch.cuda.amp import GradScaler, autocast
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models.vision_transformer import vit_b_16
from torch.utils.data import Subset, random_split
from sklearn.model_selection import GroupKFold
import numpy as np
import torch

# ---------------- CONFIG ----------------
gene_dir = "data/demo/gene_expression"   # ★ change here
img_dir  = "data/demo/images"            # ★ change here
out_dir  = "output_zinb"
cache_fp = "cache/demo_dataset.npz"      # set None to disable cache

batch_size  = 16
epochs      = 30
num_workers = 8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("💻 device:", device)
# ----------------------------------------

# ───────────────────────────────────────── Dataset ─────────
class XeniumCellDataset(Dataset):
    def __init__(self, gene_dir: str, img_dir: str, transform=None, cache_path=None):
        self.gene_dir, self.img_dir = Path(gene_dir), Path(img_dir)
        self.transform = transform or T2.Compose([
            T2.Resize((224, 224), antialias=True),
            T2.RandomHorizontalFlip(),
            T2.ColorJitter(hue=.05, saturation=.05),
            T2.ToDtype(torch.float32, scale=True),
        ])
        if cache_path and Path(cache_path).exists():
            self._load_cache(cache_path)
        else:
            self._prepare()
            if cache_path:
                self._save_cache(cache_path)

    # ---------- cache I/O ----------
    def _save_cache(self, fp):
        Path(fp).parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(fp,
            data_pairs=np.array(self.data_pairs, dtype=object),
            shared_genes=self.shared_genes,
            cell_exprs=self.cell_exprs.astype(np.float16),
            logfc_all=self.logfc_all.astype(np.float16),
            slide_ids=np.array(self.slide_ids, dtype=object))
        print(f"💾 Cached dataset → {fp}")

    def _load_cache(self, fp):
        z = np.load(fp, allow_pickle=True)
        self.data_pairs   = z["data_pairs"].tolist()
        self.shared_genes = z["shared_genes"].tolist()
        self.cell_exprs   = z["cell_exprs"]
        self.logfc_all    = z["logfc_all"]
        self.slide_ids    = z["slide_ids"].tolist()
        print(f"⚡ Loaded cached dataset from {fp}")

    # ---------- build dataset ----------
    def _prepare(self):
        expr_files = sorted([p for p in self.gene_dir.iterdir() if p.suffix == ".h5ad"])
        if not expr_files:
            raise RuntimeError(f"No .h5ad under {self.gene_dir}")

        tmp, shared = [], None
        for fp in tqdm(expr_files, desc="🔍 Reading h5ad"):
            ad = sc.read_h5ad(fp)
            shared = set(ad.var_names) if shared is None else shared & set(ad.var_names)
            tmp.append((fp.stem, ad))
        self.shared_genes = sorted(shared)
        print(f"✅ Shared genes: {len(self.shared_genes)}")

        self.data_pairs, self.slide_ids = [], []
        expr_list, coord_list = [], []
        for slide_id, ad in tqdm(tmp, desc="🖼️  Collecting pairs"):
            img_folder = self.img_dir / slide_id
            if not img_folder.is_dir():
                continue
            ad = ad[:, self.shared_genes]
            X = ad.X.toarray() if hasattr(ad.X, "toarray") else ad.X
            coords = ad.obs[["x_centroid", "y_centroid"]].values

            for i, cell_id in enumerate(ad.obs_names):
                img_fp = img_folder / f"{cell_id}.png"
                if img_fp.exists():
                    self.data_pairs.append((str(img_fp), len(expr_list)+i))
                    self.slide_ids.append(slide_id)

            expr_list.append(X); coord_list.append(coords)

        if not expr_list:
            raise RuntimeError("❌ No matching image-gene pairs found.")
        self.cell_exprs  = np.concatenate(expr_list, axis=0).astype(np.float32)
        self.cell_coords = np.concatenate(coord_list, axis=0).astype(np.float32)

        print("🔗 Pre-computing neighbor logFC ...")
        nn_model = NearestNeighbors(n_neighbors=6).fit(self.cell_coords)
        _, knn_idx = nn_model.kneighbors(self.cell_coords)
        neigh_mean = self.cell_exprs[knn_idx[:, 1:]].mean(axis=1)
        self.logfc_all = np.log2((self.cell_exprs + 1e-3)/(neigh_mean + 1e-3)).astype(np.float32)

    # ---------- torch dataset ----------
    def __len__(self): return len(self.data_pairs)
    def __getitem__(self, idx):
        img_fp, expr_idx = self.data_pairs[idx]
        img = self.transform(read_image(img_fp))
        feat = np.concatenate([self.cell_exprs[expr_idx], self.logfc_all[expr_idx]]).astype(np.float32)
        return img, torch.from_numpy(feat)

# ───────────────────────────────────────── Loss & Model (same as before) ─
def zinb_nll(x, mu, theta, pi, eps=1e-8):
    log_theta_mu_eps = torch.log(theta + mu + eps)
    nb = (torch.lgamma(theta + x) - torch.lgamma(theta) - torch.lgamma(x + 1)
          + theta * (torch.log(theta + eps) - log_theta_mu_eps)
          + x * (torch.log(mu + eps) - log_theta_mu_eps))
    zero = (x < 1e-8)
    nll = torch.where(zero,
                      -(torch.log(pi + (1.0 - pi) * torch.exp(nb) + eps)),
                      -(torch.log(1.0 - pi + eps) + nb))
    return nll.mean()

def full_loss(i_emb, g_emb, zinb, gene_input, t_img, t_gen, w_align=0.1, w_zinb=0.5):
    B = i_emb.size(0)
    lbl = torch.arange(B, device=i_emb.device)
    c_loss = (F.cross_entropy(i_emb @ g_emb.T / t_img, lbl) +
              F.cross_entropy(g_emb @ i_emb.T / t_gen, lbl)) / 2
    a_loss = F.mse_loss(i_emb, g_emb)
    G = gene_input.size(1) // 2
    z_loss = zinb_nll(gene_input[:, :G], *zinb)
    total = c_loss + w_align*a_loss + w_zinb*z_loss
    return total, c_loss.item(), a_loss.item(), z_loss.item()

class GeneTransformerEncoder(nn.Module):
    def __init__(self, gene_dim, embed_dim=768, heads=4):
        super().__init__()
        self.value_proj = nn.Linear(2, embed_dim, bias=False)
        self.gene_id_emb = nn.Embedding(gene_dim, embed_dim)
        self.pos_emb = nn.Parameter(torch.randn(1, gene_dim, embed_dim))
        enc_layer = nn.TransformerEncoderLayer(embed_dim, heads, 1024, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, 2)
    def forward(self, x):
        B,D=x.shape; G=D//2; x=x.view(B,G,2)
        tok = self.value_proj(x)+self.gene_id_emb.weight[:G]+self.pos_emb[:,:G]
        return self.transformer(tok).mean(1)

class Image2Transcripts(nn.Module):
    def __init__(self, gene_dim, embed_dim=768, vit_weights="IMAGENET1K_V1"):
        super().__init__()
        vit = vit_b_16(weights=vit_weights); vit.heads=nn.Identity()
        self.image_encoder=vit; self.image_proj=nn.Linear(768,embed_dim,bias=False)
        self.gene_encoder=GeneTransformerEncoder(gene_dim,embed_dim)
        self.gene_proj=nn.Linear(embed_dim,embed_dim,bias=False)
        self.zinb_head=nn.Linear(embed_dim,gene_dim*3)
        self.t_img=nn.Parameter(torch.tensor(0.07)); self.t_gen=nn.Parameter(torch.tensor(0.07))
        self.gene_dim=gene_dim
    def _softplus(self,x): return F.softplus(x)+1e-4
    def _split(self,z):
        z=z.view(z.size(0),3,self.gene_dim)
        return self._softplus(z[:,0]),self._softplus(z[:,1]),torch.sigmoid(z[:,2])
    def forward(self,img,gene):
        i=F.normalize(self.image_proj(self.image_encoder(img)),dim=-1)
        g_raw=self.gene_encoder(gene); g=F.normalize(self.gene_proj(g_raw),dim=-1)
        return i,g,self._split(self.zinb_head(g_raw))

# ───────────────────────────────────────── Train ──────────
def train(model, tr_ld, vl_ld, device, epochs, out_dir, lr=3e-4):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    opt = torch.optim.AdamW(model.parameters(), lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=epochs)
    scaler = GradScaler(); best = float("inf")

    logf = out / "train_log.csv"
    with open(logf, "w") as f:
        f.write("epoch,train,val,train_c,val_c,train_a,val_a,train_z,val_z\n")

    for ep in range(1, epochs + 1):
        # -------- TRAIN --------
        model.train(); met = np.zeros(4)
        for img, g in tqdm(tr_ld, desc=f"🚂 Train E{ep:03d}", leave=False):
            img, g = img.to(device, non_blocking=True), g.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast():
                loss, c, a, z = full_loss(*model(img, g), g, model.t_img, model.t_gen)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            met += np.array([loss.item(), c, a, z])
        scheduler.step()

        # -------- VAL --------
        model.eval(); met_v = np.zeros(4)
        with torch.no_grad(), autocast():
            for img, g in tqdm(vl_ld, desc=f"🧪 Val   E{ep:03d}", leave=False):
                img, g = img.to(device, non_blocking=True), g.to(device, non_blocking=True)
                loss, c, a, z = full_loss(*model(img, g), g, model.t_img, model.t_gen)
                met_v += np.array([loss.item(), c, a, z])

        # -------- LOG --------
        met   /= len(tr_ld); met_v /= len(vl_ld)
        with open(logf, "a") as f:
            f.write(f"{ep},{met[0]:.4f},{met_v[0]:.4f},{met[1]:.4f},{met_v[1]:.4f},"
                    f"{met[2]:.4f},{met_v[2]:.4f},{met[3]:.4f},{met_v[3]:.4f}\n")

        print(f"E{ep:03d}  L {met[0]:.3f}/{met_v[0]:.3f}  "
              f"C {met[1]:.3f}/{met_v[1]:.3f}  "
              f"A {met[2]:.3f}/{met_v[2]:.3f}  "
              f"Z {met[3]:.3f}/{met_v[3]:.3f}")

        if met_v[0] < best:
            best = met_v[0]
            torch.save(model.state_dict(), out / "best_model.pt")
            print("   ✔️  saved best")

# ───────────────────────────────────────── Main ───────────
if __name__=="__main__":
    tfm = T2.Compose([
        T2.Resize((224,224), antialias=True),
        T2.RandomHorizontalFlip(),
        T2.ColorJitter(hue=.05, saturation=.05),
        T2.ToDtype(torch.float32, scale=True),
    ])
    ds = XeniumCellDataset(gene_dir, img_dir, transform=tfm, cache_path=cache_fp)
    ds = XeniumCellDataset(gene_dir, img_dir, transform=tfm, cache_path=cache_fp)

    unique_slides = np.unique(ds.slide_ids)
    if len(unique_slides) > 1:
        # -------- slide-level GroupKFold --------
        n_splits = min(5, len(unique_slides))          # 不能多于 slide 数
        gkf = GroupKFold(n_splits=n_splits)
        train_idx, val_idx = next(gkf.split(
            np.arange(len(ds)), groups=ds.slide_ids))
        tr_ds, vl_ds = Subset(ds, train_idx), Subset(ds, val_idx)
        print(f"🔀 GroupKFold ({n_splits}-fold)  train={len(tr_ds)}  val={len(vl_ds)}")
    else:
        # -------- 单 slide ⇒ 随机 80/20 --------
        train_len = int(0.8 * len(ds)); val_len = len(ds) - train_len
        tr_ds, vl_ds = random_split(
            ds, [train_len, val_len],
            generator=torch.Generator().manual_seed(42))
        print(f"🔀 Random split  train={train_len}  val={val_len}")

    tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,
                       num_workers=num_workers, pin_memory=True,
                       persistent_workers=True, prefetch_factor=4)
    vl_ld = DataLoader(vl_ds, batch_size=batch_size, shuffle=False,
                       num_workers=num_workers, pin_memory=True,
                       persistent_workers=True, prefetch_factor=4)

    model = Image2Transcripts(gene_dim=len(ds.shared_genes)).to(device)
    train(model, tr_ld, vl_ld, device, epochs, out_dir)


💻 device: cpu
⚡ Loaded cached dataset from cache/demo_dataset.npz
⚡ Loaded cached dataset from cache/demo_dataset.npz
🔀 Random split  train=12616  val=3155


/tmp/ipykernel_25300/1428496124.py:184: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(); best = float("inf")
/archive/DPDS/Xiao_lab/shared/jia_yao/envs/image2transcripts/lib/python3.9/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
🚂 Train E001:   0%|          | 0/789 [00:00<?, ?it/s]/tmp/ipykernel_25300/1428496124.py:196: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/archive/DPDS/Xiao_lab/shared/jia_yao/envs/image2transcripts/lib/python3.9/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


KeyboardInterrupt: 